In [2]:
import torch
from transformers import BertForQuestionAnswering
from transformers import BertTokenizer


model_name = "pborchert/BusinessBERT"

#Model
model = BertForQuestionAnswering.from_pretrained(model_name)

#Tokenizer
tokenizer = BertTokenizer.from_pretrained(model_name)


config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/437M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForQuestionAnswering LOAD REPORT from: pborchert/BusinessBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
qa_outputs.bias                            | MISSING    | 
qa_outputs.weight                          | MISS

model.safetensors:   0%|          | 0.00/437M [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

In [72]:
question = "to what extent does the company use industrial digital technologies?"

context = "the company uses a wide range of industrial digital technologies including some internal data collection and analysis systems"
# TODO adapt code so that it can handle longer contexts which need to be split into chunks
#, they have internal data management systems incl CRM and full suite MRP and have developed their application of this to now provide full client journey management using the system. They have started to explore the potential to include automated testing systems in the manufacturing process, although this is still in the early adoption phase they see this as a likely integral technology moving forwards. They have also implemented GPS tracking of product/WIP in certain areas of their factory and see expanding this out across the entire site as well as increasing use of robots/cobots as crucial next steps in their evolution."

inputs = tokenizer(
    question,
    context,
    max_length=100,
    truncation="only_second",
    stride=50,
    return_overflowing_tokens=True,
    )

input_ids = inputs['input_ids'][0]
print('The input has a total of {:} tokens.'.format(len(input_ids)))

The input has a total of 32 tokens.


In [73]:
for ids in inputs["input_ids"]:
    print(tokenizer.decode(ids))

[CLS] to what extent does the company use industrial digital technologies? [SEP] the company uses a wide range of industrial digital technologies including some internal data collection and analysis systems [SEP]


In [74]:
context_tokens = inputs['input_ids'][0][inputs['input_ids'][0].index(3)+1: -1]

In [75]:
tokens = tokenizer.convert_ids_to_tokens(input_ids)

# For each token and its id...
for token, id in zip(tokens, input_ids):

    # If this is the [SEP] token, add some space around it to make it stand out.
    if id == tokenizer.sep_token_id:
        print('')

    # Print the token string and its ID in two columns.
    print('{:<12} {:>6,}'.format(token, id))

    if id == tokenizer.sep_token_id:
        print('')

[CLS]             0
to            1,894
what          2,496
extent        2,484
does          2,317
the           1,891
company       1,921
use           2,017
industrial    2,567
digital       3,057
technologies  2,547
?             1,035

[SEP]             3

the           1,891
company       1,921
uses          2,845
a             1,043
wide          3,092
range         2,519
of            1,892
industrial    2,567
digital       3,057
technologies  2,547
including     1,983
some          2,117
internal      2,472
data          2,042
collection    2,841
and           1,893
analysis      2,089
systems       2,176

[SEP]             3



In [76]:
start_scores, end_scores = model(torch.tensor(inputs['input_ids']),
                                 token_type_ids=torch.tensor(inputs['token_type_ids']),
                                 return_dict=False)

In [77]:
answer_start = torch.argmax(start_scores)
answer_end = torch.argmax(end_scores)
answer = tokenizer.decode(context_tokens[answer_start:answer_end+1])

print('Answer: "' + answer + '"')

Answer: "a wide range of industrial digital technologies including some internal data collection and analysis systems"
